# 02 평가 — pass@1 + LLM judge (CPU)

**런타임**: CPU면 충분합니다(GPU 불필요). 01이 끝난 뒤 실행.

**순서와 격리**
1. **pass@1을 먼저** 돌립니다. 이 단계에서는 API 키를 읽지 않습니다. 생성 코드는 임시 폴더에서 `python -I`, 환경변수 최소화, 메모리 2GB 제한, 10초 타임아웃으로 실행됩니다. (완전한 샌드박스는 아님 — 보고서에는 "일회용 Colab 런타임의 별도 프로세스에서, API 키를 읽기 전에 실행"이라고 적으세요.)
2. 정답 코드 164개가 전부 통과하는지 먼저 확인합니다(채점기 검증). 하나라도 실패하면 멈춥니다.
3. 그다음 judge. judge는 pass@1 결과를 보지 않습니다(둘은 03에서만 합쳐짐). judge에는 생성 모델 이름도 주지 않습니다(블라인드).

**judge 실행 모드** (`JUDGE_MODE`)
- `"smoke"`: judge별 1건 호출해서 모델 ID·파싱·응답 잘림을 확인. 저장 안 함
- `"sample"`: 앞쪽 `SAMPLE_TASKS`문제 × 전 키 × 전 judge를 채점해 저장하고, 호출당 비용으로 전체 비용 추정
- `"full"`: 나머지 전부 + 안정성 측정용 재호출(`REPEAT_N`건). sample에서 채점한 건은 건너뜀

**저장** (`MyDrive/code_eval_v4/<RUN_NAME>/`)
- `canonical_check.json`, `pass_results.jsonl`
- `judge_results.jsonl` — 호출 1건당 1줄(원문 응답, 파싱 결과, 상태, 반환된 모델 버전 문자열, 토큰 사용량)
- 상태값: `ok` / `parse_error`(judge가 형식을 못 지킴 — 재시도하지 않고 그대로 집계) / `truncated`(출력 한도에서 잘림) / `api_error`(다시 실행하면 재시도)

In [ ]:
!pip install -q -U datasets anthropic openai google-genai

In [ ]:
RUN_NAME = "smoke"        # 01과 같은 값. "smoke" → "full"

import os, json, glob
from google.colab import drive
from datasets import load_dataset

drive.mount("/content/drive")
RUN_DIR = f"/content/drive/MyDrive/code_eval_v4/{RUN_NAME}"
GEN_DIR = f"{RUN_DIR}/generations"
PROBLEMS = {p["task_id"]: p for p in load_dataset("openai/openai_humaneval", split="test")}

In [ ]:
# 생성 결과 로드 + 실행 채점 함수
import subprocess, sys, tempfile, resource
from concurrent.futures import ThreadPoolExecutor
from collections import Counter

def read_jsonl(path):
    if not os.path.exists(path):
        return []
    with open(path) as f:
        return [json.loads(l) for l in f if l.strip()]

def load_generations(gen_dir):
    gens = {}
    for path in sorted(glob.glob(f"{gen_dir}/*.jsonl")):
        key = os.path.basename(path)[:-len(".jsonl")]
        seen, recs = set(), []
        for r in read_jsonl(path):
            if r["task_id"] not in seen:          # 중복 레코드는 첫 번째만 사용
                seen.add(r["task_id"]); recs.append(r)
        gens[key] = recs
    return gens

GENS = load_generations(GEN_DIR)
print({k: len(v) for k, v in GENS.items()})

TIMEOUT_S = 10
MEM_LIMIT = 2 * 1024 ** 3

def _limit_resources():
    resource.setrlimit(resource.RLIMIT_AS, (MEM_LIMIT, MEM_LIMIT))

def run_program(code, timeout=TIMEOUT_S):
    """별도 프로세스에서 실행. 반환 (통과 여부, 에러 유형, 마지막 에러 줄)"""
    with tempfile.TemporaryDirectory() as d:
        path = os.path.join(d, "prog.py")
        with open(path, "w") as f:
            f.write(code)
        env = {"PATH": os.environ.get("PATH", ""), "PYTHONDONTWRITEBYTECODE": "1", "PYTHONHASHSEED": "0"}
        try:
            r = subprocess.run([sys.executable, "-I", path], cwd=d, env=env, capture_output=True,
                               text=True, timeout=timeout, preexec_fn=_limit_resources)
        except subprocess.TimeoutExpired:
            return False, "Timeout", ""
    if r.returncode == 0:
        return True, "", ""
    lines = [l for l in r.stderr.strip().splitlines() if l.strip()]
    last = lines[-1] if lines else f"exit {r.returncode}"
    etype = last.split(":")[0].strip() if not last.startswith("exit ") else "Exit"
    return False, etype, last[:300]

def test_program(exec_code, problem):
    return exec_code + "\n\n" + problem["test"] + f"\n\ncheck({problem['entry_point']})\n"

def run_many(codes, workers=8):
    with ThreadPoolExecutor(max_workers=workers) as ex:
        return list(ex.map(run_program, codes))

In [ ]:
# 채점기 검증: 정답 코드(canonical solution) 164개가 전부 통과해야 함
ids = list(PROBLEMS)
res = run_many([test_program(PROBLEMS[t]["prompt"] + PROBLEMS[t]["canonical_solution"], PROBLEMS[t]) for t in ids])
failed = [(t, e, m) for t, (ok, e, m) in zip(ids, res) if not ok]
json.dump({"n": len(ids), "n_pass": len(ids) - len(failed), "failed": failed},
          open(f"{RUN_DIR}/canonical_check.json", "w"), indent=2)
print(f"정답 코드 통과 {len(ids) - len(failed)}/{len(ids)}")
assert not failed, f"채점 환경 문제: {failed[:5]}"

In [ ]:
# pass@1 (greedy 1회 생성이므로 pass@1 = 통과 문제 수 / 전체)
pass_rows = []
for key, recs in GENS.items():
    res = run_many([test_program(r["exec_code"], PROBLEMS[r["task_id"]]) for r in recs])
    for r, (ok, etype, msg) in zip(recs, res):
        pass_rows.append({"task_id": r["task_id"], "model_key": key, "passed": ok,
                          "error_type": etype, "error_msg": msg})
    n_pass = sum(ok for ok, _, _ in res)
    print(f"[{key}] pass@1 = {n_pass}/{len(recs)} = {n_pass/max(len(recs),1):.3f}")
    fails = Counter(e for ok, e, _ in res if not ok)
    if fails:
        print("   실패 유형:", dict(fails.most_common()))

with open(f"{RUN_DIR}/pass_results.jsonl", "w") as f:
    for row in pass_rows:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")
print("저장:", f"{RUN_DIR}/pass_results.jsonl")

## LLM judge

- 아래 설정 셀의 `JUDGE_MODE`를 `"smoke"` → `"sample"` → `"full"` 순서로 바꿔 가며 **설정 셀부터** 다시 실행하세요.
- 모델 ID가 틀리면 smoke에서 바로 보입니다. 맨 아래 "사용 가능한 모델 목록" 셀로 실제 ID를 확인할 수 있습니다.
- 가격은 이전 노트북에 적어 둔 값(제3자 정리 기준)입니다. 공식 가격 페이지에서 확인 후 고치세요. 비용 추정에만 쓰입니다.
- judge 등급을 비교하려면(중간 등급 vs 상위 등급) `JUDGES`에 상위 모델을 추가하고 `sample` 모드로 돌린 뒤 03에서 비교하면 됩니다.

In [ ]:
# ============================================================
# judge 설정 — 여기만 바꾸면 됩니다
# ============================================================
JUDGE_MODE = "smoke"      # "smoke" → "sample" → "full"
SAMPLE_TASKS = 5          # sample 모드: 앞쪽 몇 문제를 채점할지 (× 키 수 × judge 수)
REPEAT_N = 20             # full 모드: judge마다 무작위 (키, 문제) 몇 건을 한 번 더 호출해 흔들림 측정
N_WORKERS = 4             # 동시 호출 수. 429 에러가 많으면 낮추기
SEED = 0
JUDGE_KEYS = None         # None이면 생성 키 전부 채점. 일부만: ["starcoder2-3b", "qwen2.5-7b-instruct"]

# price: (입력, 출력) 달러/100만 토큰 — 실행 전 공식 가격 확인. kwargs: API에 그대로 넘길 추가 인자
JUDGES = {
    "claude-sonnet-5-5": {"provider": "anthropic", "model_id": "claude-sonnet-5-5", "max_tokens": 4000,
                          "price": (2.0, 10.0), "kwargs": {}},
    "gpt-5.6-terra":     {"provider": "openai", "model_id": "gpt-5.6-terra", "max_tokens": 4000,
                          "price": (2.0, 12.0), "kwargs": {}},
    "gemini-3.1-pro":    {"provider": "google", "model_id": "gemini-3.1-pro", "max_tokens": 8000,
                          "price": (2.0, 12.0), "kwargs": {}},
    "gemini-3.8-flash":  {"provider": "google", "model_id": "gemini-3.8-flash", "max_tokens": 8000,
                          "price": (0.75, 3.75), "kwargs": {}},
}
# temperature를 고정하고 싶으면 kwargs에 {"temperature": 0} — 추론 모델은 거부할 수 있으니 smoke로 확인

# ============================================================
from google.colab import userdata

def _secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None

API_KEYS = {"anthropic": _secret("ANTHROPIC_API_KEY"), "openai": _secret("OPENAI_API_KEY"),
            "google": _secret("GOOGLE_API_KEY")}
used = {c["provider"] for c in JUDGES.values()}
print("키 로드:", {p: bool(API_KEYS[p]) for p in used})
JUDGE_PATH = f"{RUN_DIR}/judge_results.jsonl"

In [ ]:
# judge 함수: 프롬프트, 파싱, API 호출, 작업 실행
import re, time, random, threading, datetime

CRITERIA = ["correctness", "readability", "efficiency", "robustness"]
RUBRIC_VERSION = "r2"     # r1(이전 노트북): 1/3/5점 기준만. r2: 5단계 기준 + 근거 먼저 + 구현만 평가
RUBRIC_PROMPT = """당신은 Python 코드 품질을 평가하는 전문가입니다. 아래 [문제]는 함수 시그니처와 docstring이고, [코드]는 이 문제에 대해 작성된 구현입니다.
코드를 실행할 수 없고 테스트 케이스도 주어지지 않습니다. 코드를 읽고 판단하세요.
문제에 주어진 시그니처와 docstring 자체는 평가하지 말고, 구현 부분을 평가하세요.

[평가 항목]
- correctness(정확성): docstring의 요구사항과 예시를 허용되는 모든 입력에서 올바르게 만족하는가
- readability(가독성): 이름, 구조, 흐름이 읽고 이해하기 쉬운가
- efficiency(효율성): 문제 규모에 비해 불필요한 계산이나 자료구조가 없는가
- robustness(견고성): 경계값, 빈 입력 등 docstring이 허용하는 입력에서 안전하게 동작하는가

[점수 기준] 각 항목 1~5 정수
- 5: 해당 항목에서 결함이 없음
- 4: 사소한 결함이 있음
- 3: 기본은 충족하나 뚜렷한 개선 여지가 있음
- 2: 중요한 결함이 있음
- 1: 심각한 결함이 있음 (예: correctness에서 핵심 요구를 만족하지 못함)

[문제]
<<PROBLEM>>

[코드]
```python
<<CODE>>
```

각 항목마다 근거를 먼저 쓰고 점수를 매기세요. 반드시 아래 형식의 JSON 하나로만 답하고, 다른 텍스트는 쓰지 마세요.
{"correctness": {"reason": "...", "score": 1~5}, "readability": {"reason": "...", "score": 1~5}, "efficiency": {"reason": "...", "score": 1~5}, "robustness": {"reason": "...", "score": 1~5}}
"""

def build_prompt(problem_prompt, judge_code):
    # 생성 모델 이름은 넣지 않음(블라인드). replace를 쓰는 이유: 코드 안의 중괄호가 format과 충돌하지 않게
    return RUBRIC_PROMPT.replace("<<PROBLEM>>", problem_prompt.strip()).replace("<<CODE>>", judge_code.rstrip())

def _as_score(v):
    if isinstance(v, bool):
        raise ValueError("bool")
    if isinstance(v, (int, float)) and float(v).is_integer():
        v = int(v)
    elif isinstance(v, str) and v.strip().isdigit():
        v = int(v.strip())
    else:
        raise ValueError(f"정수 아님: {v!r}")
    if not 1 <= v <= 5:
        raise ValueError(f"범위 밖: {v}")
    return v

def parse_judge(text):
    """반환: {항목: 점수} (근거는 원문 응답에 남아 있음). 실패 시 예외"""
    t = (text or "").strip()
    t = re.sub(r"^```(?:json)?\s*", "", t)
    t = re.sub(r"\s*```\s*$", "", t)
    s, e = t.find("{"), t.rfind("}")
    if s < 0 or e <= s:
        raise ValueError("JSON 없음")
    data = json.loads(t[s:e + 1])
    return {c: _as_score(data[c]["score"]) for c in CRITERIA}

# ---------------- API 호출 (반환 형식 통일) ----------------
_clients, _client_lock = {}, threading.Lock()

def client(provider):
    with _client_lock:
        if provider not in _clients:
            if provider == "anthropic":
                import anthropic
                _clients[provider] = anthropic.Anthropic(api_key=API_KEYS["anthropic"])
            elif provider == "openai":
                from openai import OpenAI
                _clients[provider] = OpenAI(api_key=API_KEYS["openai"])
            elif provider == "google":
                from google import genai
                _clients[provider] = genai.Client(api_key=API_KEYS["google"])
        return _clients[provider]

def call_anthropic(cfg, prompt):
    m = client("anthropic").messages.create(model=cfg["model_id"], max_tokens=cfg["max_tokens"],
                                            messages=[{"role": "user", "content": prompt}], **cfg["kwargs"])
    text = "".join(b.text for b in m.content if getattr(b, "type", "") == "text")
    return {"text": text, "model_returned": m.model, "stop": m.stop_reason,
            "truncated": m.stop_reason == "max_tokens",
            "usage": {"input": m.usage.input_tokens, "output": m.usage.output_tokens, "thinking": None}}

def call_openai(cfg, prompt):
    r = client("openai").chat.completions.create(model=cfg["model_id"], max_completion_tokens=cfg["max_tokens"],
                                                 messages=[{"role": "user", "content": prompt}], **cfg["kwargs"])
    c, u = r.choices[0], r.usage
    think = getattr(getattr(u, "completion_tokens_details", None), "reasoning_tokens", None)
    return {"text": c.message.content or "", "model_returned": r.model, "stop": c.finish_reason,
            "truncated": c.finish_reason == "length",
            "usage": {"input": u.prompt_tokens, "output": u.completion_tokens, "thinking": think}}

def call_google(cfg, prompt):
    from google.genai import types
    r = client("google").models.generate_content(
        model=cfg["model_id"], contents=prompt,
        config=types.GenerateContentConfig(max_output_tokens=cfg["max_tokens"], **cfg["kwargs"]))
    fr = r.candidates[0].finish_reason if r.candidates else None
    um = r.usage_metadata
    cand = getattr(um, "candidates_token_count", 0) or 0
    th = getattr(um, "thoughts_token_count", 0) or 0
    return {"text": r.text or "", "model_returned": getattr(r, "model_version", None), "stop": str(fr),
            "truncated": "MAX_TOKENS" in str(fr),
            "usage": {"input": getattr(um, "prompt_token_count", 0) or 0, "output": cand + th, "thinking": th}}

PROVIDERS = {"anthropic": call_anthropic, "openai": call_openai, "google": call_google}

def is_transient(e):
    s = str(e).lower()
    return any(w in s for w in ("429", "rate", "overload", "500", "502", "503", "504",
                                "timeout", "timed out", "unavailable", "connection"))

def call_with_retry(cfg, prompt, tries=5):
    for k in range(tries):
        try:
            return PROVIDERS[cfg["provider"]](cfg, prompt)
        except Exception as e:
            if k == tries - 1 or not is_transient(e):
                raise
            time.sleep(min(60, 2 ** (k + 1)) + random.random())

def run_job(job):
    """job = (model_key, task_id, judge_name, rep). 파싱 실패는 재시도하지 않고 그대로 기록"""
    key, tid, jname, rep = job
    cfg = JUDGES[jname]
    rec_gen = GEN_INDEX[(key, tid)]
    prompt = build_prompt(PROBLEMS[tid]["prompt"], rec_gen["judge_code"])
    base = {"model_key": key, "task_id": tid, "judge": jname, "rep": rep,
            "judge_model_id": cfg["model_id"], "rubric_version": RUBRIC_VERSION,
            "created_at": datetime.datetime.now().isoformat(timespec="seconds")}
    t0 = time.time()
    try:
        out = call_with_retry(cfg, prompt)
    except Exception as e:
        return {**base, "status": "api_error", "error": str(e)[:500], "latency_s": round(time.time() - t0, 1)}
    rec = {**base, "model_returned": out["model_returned"], "stop": out["stop"], "usage": out["usage"],
           "raw_text": out["text"], "latency_s": round(time.time() - t0, 1)}
    try:
        rec["scores"] = parse_judge(out["text"])
        rec["status"] = "ok"
    except Exception as e:
        rec["status"] = "truncated" if out["truncated"] else "parse_error"
        rec["error"] = str(e)[:300]
    return rec

def job_cost(rec):
    u, cfg = rec.get("usage"), JUDGES.get(rec["judge"])
    if not u or not cfg:
        return 0.0
    pin, pout = cfg["price"]
    return u["input"] * pin / 1e6 + u["output"] * pout / 1e6

In [ ]:
# 실행
from concurrent.futures import as_completed
from tqdm.auto import tqdm

gen_keys = JUDGE_KEYS or list(GENS)
GEN_INDEX = {(k, r["task_id"]): r for k in gen_keys for r in GENS[k]}
task_order = [t for t in PROBLEMS if any((k, t) in GEN_INDEX for k in gen_keys)]

def planned_jobs(mode):
    tasks = task_order[:SAMPLE_TASKS] if mode == "sample" else task_order
    jobs = [(k, t, j, 0) for t in tasks for k in gen_keys if (k, t) in GEN_INDEX for j in JUDGES]
    if mode == "full" and REPEAT_N:
        # 모든 judge가 같은 (키, 문제) 표본을 한 번 더 채점 → judge끼리 흔들림 비교 가능
        rep_pairs = random.Random(SEED).sample(sorted(GEN_INDEX), min(REPEAT_N, len(GEN_INDEX)))
        jobs += [(k, t, j, 1) for k, t in rep_pairs for j in JUDGES]
    return jobs

def summarize(recs, extrapolate_to=None):
    agg = {}
    for r in recs:
        a = agg.setdefault(r["judge"], Counter())
        a["n"] += 1; a[r["status"]] += 1; a["cost"] += job_cost(r)
        u = r.get("usage") or {}
        a["in"] += u.get("input", 0) or 0; a["out"] += u.get("output", 0) or 0
    print(f"{'judge':20s} {'건수':>5s} {'ok':>5s} {'파싱실패':>6s} {'잘림':>4s} {'API에러':>6s} {'호출당입력':>8s} {'호출당출력':>8s} {'비용$':>7s}")
    for j, a in agg.items():
        n = max(a["n"], 1)
        print(f"{j:20s} {a['n']:5d} {a['ok']:5d} {a['parse_error']:6d} {a['truncated']:4d} {a['api_error']:6d}"
              f" {a['in']/n:8.0f} {a['out']/n:8.0f} {a['cost']:7.2f}")
    if extrapolate_to:
        per = {j: a["cost"] / max(a["n"] - a["api_error"], 1) for j, a in agg.items()}
        total = sum(per[j] * extrapolate_to.get(j, 0) for j in per)
        print(f"\nfull 모드 전체 예상 비용: ${total:.2f}  (judge별 호출당 비용 × 남은 호출 수)")

if JUDGE_MODE == "smoke":
    k, t = next(iter(GEN_INDEX))
    print(f"스모크: {k} / {t}\n")
    for j in JUDGES:
        r = run_job((k, t, j, 0))
        print(f"[{j}] status={r['status']} | 반환 모델={r.get('model_returned')} | stop={r.get('stop')} | usage={r.get('usage')}")
        print(f"   {r.get('error', '')}")
        print(f"   응답 앞부분: {(r.get('raw_text') or '')[:300]!r}\n")
    print("전부 status=ok이고 반환 모델이 의도한 것이면 JUDGE_MODE = 'sample'로 바꿔 설정 셀부터 다시 실행")
else:
    existing = read_jsonl(JUDGE_PATH)
    done = {(r["model_key"], r["task_id"], r["judge"], r["rep"]) for r in existing if r["status"] != "api_error"}
    jobs = [jb for jb in planned_jobs(JUDGE_MODE) if jb not in done]
    random.Random(SEED).shuffle(jobs)        # 순서 효과·한 모델에 몰린 호출 방지
    print(f"모드 {JUDGE_MODE}: 실행 예정 {len(jobs)}건 (이미 완료 {len(done)}건, api_error는 재시도)")

    lock = threading.Lock()
    with open(JUDGE_PATH, "a") as f, ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
        futs = [ex.submit(run_job, jb) for jb in jobs]
        for fut in tqdm(as_completed(futs), total=len(futs), desc="judge"):
            rec = fut.result()
            with lock:
                f.write(json.dumps(rec, ensure_ascii=False) + "\n"); f.flush()

    # 같은 작업의 최신 레코드만 남겨서 요약 (api_error 후 재시도 성공한 경우 등)
    latest = {}
    for r in read_jsonl(JUDGE_PATH):
        latest[(r["model_key"], r["task_id"], r["judge"], r["rep"])] = r
    recs = list(latest.values())
    remaining = None
    if JUDGE_MODE == "sample":
        full_done = {jk for jk, r in latest.items() if r["status"] != "api_error"}
        left = [jb for jb in planned_jobs("full") if jb not in full_done]
        remaining = Counter(j for _, _, j, _ in left)
    summarize(recs, remaining)
    errs = [r for r in recs if r["status"] == "api_error"]
    if errs:
        print(f"\napi_error {len(errs)}건 — 이 셀을 다시 실행하면 그것만 재시도. 예: {errs[0]['error'][:200]}")

### (필요할 때만) 사용 가능한 모델 목록
judge 설정 셀과 함수 셀을 실행한 뒤 사용. smoke에서 "model not found" 류의 에러가 나면 이 셀로 실제 ID를 확인하고 `JUDGES`의 `model_id`를 고치세요.

In [ ]:
KEYWORDS = ["claude", "gpt-5", "gpt-6", "gemini"]
for provider in sorted(used):
    try:
        if provider == "anthropic":
            names = [m.id for m in client("anthropic").models.list(limit=100)]
        elif provider == "openai":
            names = [m.id for m in client("openai").models.list()]
        else:
            names = [m.name for m in client("google").models.list()]
        names = sorted(n for n in names if any(k in n for k in KEYWORDS))
        print(f"[{provider}] {len(names)}개\n  " + "\n  ".join(names))
    except Exception as e:
        print(f"[{provider}] 목록 조회 실패: {e}")